In [ ]:
!date

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from concurrent.futures import ProcessPoolExecutor, as_completed
from tqdm import tqdm

from scipy.ndimage import gaussian_filter
import re
from sklearn.cluster import KMeans

In [ ]:
projdir = '/u/project/cluo/terencew/igvf/2023_YR2/snmCT/pseudobulk_allc'
donors = sorted(list(np.loadtxt(f'{projdir}/txt/donors.txt', dtype=str)))
tmp_order = ['Start', 'Sendai',  'Delayed', 'Fail1','Inter1',  'Inter2', 'Fail2', 'IPS']
clusts = [f'{x}_{y}' for x,y in zip(np.repeat(tmp_order, len(donors)), np.tile(donors, len(tmp_order)))]

In [ ]:
rev_class = ['Start_1', 'Start_2', 'Sendai_1', 'Sendai_2',
     'Delayed_1', 'Delayed_2', 'Delayed_3', 'Fail1_1', 'Fail1_3', 'Fail1_4',
      'Inter1_1', 'Inter1_2', 'Inter1_3', 'Inter1_5', 'Inter2_1',
     'Inter2_2', 'Inter2_3', 'Fail2_2', 'Fail2_3', 'Fail2_5',
      'IPS_2', 'IPS_3']

### load enhancer methylation

In [ ]:
peaks_path = '/u/project/cluo/terencew/igvf/2023_YR2/multiome/atac/peaks/chromhmm/justin.chromhmm.dedup.bed.gz'
peaks = pd.read_csv(peaks_path, sep='\t', header=None)
peaks.index = [f'{x}_{y}' for x,y in zip(peaks[0], peaks[1])]
peaks.shape

In [ ]:
peaks.head()

In [ ]:
def process_bed(s):
    peaks_mc = pd.read_csv(f'{projdir}/bed/features/rev_class/{s}.peaks_CGN-Both.full.bed.gz', sep='\t', header=None)
    peaks_mc.index = [f'{x}_{y}' for x,y in zip(peaks_mc[0], peaks_mc[1])]
    return peaks_mc

with ProcessPoolExecutor() as executor:
    results = list(tqdm(executor.map(process_bed, rev_class), total=len(rev_class)))

In [ ]:
results[0].head()

In [ ]:
results[0].shape

In [ ]:
mc_results = [x[15] / x[16] for x in results]
peaks_df = pd.concat(mc_results, axis=1)
peaks_df.columns = rev_class

for s in rev_class:
    peaks_df[s].fillna(peaks_df[s].mean())
    
peaks_df = peaks_df[~peaks_df.index.duplicated()]
peaks_df.shape

In [ ]:
#                'Sendai_1', 'Sendai_2',
#                'Delayed_1', 'Delayed_2', 'Delayed_3',
#               'Fail1_1', 'Fail1_2', 'Fail1_3', 'Fail1_4',
#               'Inter1_1', 'Inter1_2', 'Inter1_3', 'Inter1_4','Inter1_5',
#               'Inter2_1', 'Inter2_2', 'Inter2_3', 'Inter2_4',
#               'Fail2_2', 'Fail2_3', 'Fail2_4', 'Fail2_5',
#                 'IPS_1', 'IPS_2', 'IPS_3']

In [ ]:
annots = peaks[14].unique()
annots

In [ ]:
a = 'Enhancer'
annot_peaks = peaks[peaks[14] == a]
annot_peaks.head()

In [ ]:
peak_results = []
for s in rev_class:
    tmp_peaks = annot_peaks[annot_peaks[12] == s]
    mean_mcg = peaks_df.reindex(tmp_peaks.index).mean(axis=0).values
    peak_results.append(mean_mcg)
    
mcg_df = pd.DataFrame(np.vstack(peak_results))
mcg_df.columns = rev_class
mcg_df.index = rev_class
mcg_df = mcg_df.dropna()
mcg_df.shape

In [ ]:
mcg_df.head()

In [ ]:
fig, axes = plt.subplots(1, figsize=(8,6))
ax = sns.heatmap(mcg_df, vmin=0.3, vmax=0.7, cmap='coolwarm')
ax.set_title(a)

### super basic - what proportion of IPS peaks are methylated/unmethylated?

In [ ]:
annot_peaks[14].value_counts()

In [ ]:
s = 'IPS'
ips_enh = annot_peaks[annot_peaks[13] == 'IPS']
ips_enh.shape

In [ ]:
select = ['Start_1', 'Start_2', 'Inter2_3', 'Fail2_3', 'IPS_2', 'IPS_3']
tmp_df = peaks_df.reindex(ips_enh.index)[select].dropna()
tmp_df.shape

In [ ]:
k = 6
kmeans = KMeans(n_clusters=k)
kmeans.fit(tmp_df)
k_clusters = kmeans.predict(tmp_df)
tmp_df['cluster'] = k_clusters
cluster_sizes = tmp_df['cluster'].value_counts().sort_index().values
tmp_df = tmp_df.sort_values(by=['cluster']).iloc[:,:-1]

boundaries = np.cumsum(cluster_sizes)[:-1]

ax = sns.heatmap(tmp_df, cmap='coolwarm', vmin=0.3, vmax=0.7)

for b in boundaries:
    ax.hlines(b, *ax.get_xlim(), colors='black', linewidth=2)

plt.show()

In [ ]:
mask = tmp_df['Fail2_3'] > 0.5
mask.sum() / tmp_df.shape[0]

In [ ]:
outdir = '/u/project/cluo/terencew/igvf/2023_YR2/final_figures/csv/figure3'
tmp_df.to_csv(f'{outdir}/rev_class_chromhmm_peaks_mcg.csv', sep='\t')

In [ ]:
!date